# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd
import numpy as np

def load_and_standardize_data(urls: list[str]) -> pd.DataFrame:
    """
    Carga múltiples archivos CSV desde URLs y estandariza los nombres de las columnas.
    """
    dataframes = []
    
    for url in urls:
        df = pd.read_csv(url)
        # Estandarizamos nombres: minúsculas y reemplazo de espacios por guiones bajos
        df.columns = [col.lower().replace(" ", "_") for col in df.columns]
        dataframes.append(df)
        
    # Concatenamos todos los DataFrames
    return pd.concat(dataframes, ignore_index=True)

def clean_insurance_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia y formatea el dataset de seguros combinado siguiendo reglas de negocio.
    """
    # 1. Limpieza de la columna 'st' (State)
    state_mapping = {
        'Cali': 'California',
        'AZ': 'Arizona',
        'WA': 'Washington'
    }
    df['st'] = df['st'].replace(state_mapping)
    
    # 2. Limpieza de la columna 'gender'
    # Basado en el análisis previo de valores únicos ('Femal', 'female', 'F')
    gender_mapping = {
        'Femal': 'F',
        'female': 'F',
        'Male': 'M',
        'male': 'M'
    }
    df['gender'] = df['gender'].replace(gender_mapping)
    
    # 3. Formateo de tipos de datos
    # La columna 'customer' es un ID alfanumérico, nos aseguramos que sea string
    df['customer'] = df['customer'].astype(str)
    
    # 4. Manejo de valores nulos
    # Eliminamos duplicados exactos si existen tras la combinación
    df = df.drop_duplicates()
    
    return df

# --- Ejecución del proceso ---

urls_seguros = [
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"
]

# Paso 1: Carga y combinación
combined_df = load_and_standardize_data(urls_seguros)

# Paso 2: Limpieza profunda
final_df = clean_insurance_data(combined_df)

# Verificación de resultados
print(f"Dimensiones finales del dataset: {final_df.shape}")
print(final_df['st'].unique())
print(final_df['gender'].unique())

Dimensiones finales del dataset: (9135, 12)
<StringArray>
['Washington', 'Arizona', 'Nevada', 'California', 'Oregon', nan]
Length: 6, dtype: str
<StringArray>
[nan, 'F', 'M']
Length: 3, dtype: str


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [3]:
import pandas as pd
import numpy as np

def initialize_marketing_data(url: str) -> pd.DataFrame:
    """
    Loads the marketing dataset and standardizes column names 
    for better accessibility in code.
    """
    df = pd.read_csv(url)
    
    # Standardizing columns: lower case and underscores
    df.columns = [col.lower().replace(" ", "_") for col in df.columns]
    
    # Dropping any potential fully empty rows or columns 
    # that often appear in web-hosted CSVs
    df = df.dropna(how='all')
    
    return df

# URL for the clean dataset
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df_marketing = initialize_marketing_data(data_url)

# Displaying structural info
print(f"Columns found: {df_marketing.columns.tolist()}")

Columns found: ['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']


In [4]:
import pandas as pd

# 1. Create a pivot table summarizing total revenue (CLV) per sales channel
# We use 'sum' to get the total revenue and 'round' for the 2 decimal points
sales_channel_revenue = df_marketing.pivot_table(
    index='sales_channel', 
    values='customer_lifetime_value', 
    aggfunc='sum'
).round(2)

# 2. Sort the values to make the top-performing channel immediately obvious
sales_channel_revenue = sales_channel_revenue.sort_values(
    by='customer_lifetime_value', 
    ascending=False
)

# Display the resulting table
print("Total Revenue by Sales Channel:")
print(sales_channel_revenue)

Total Revenue by Sales Channel:
               customer_lifetime_value
sales_channel                         
Agent                      33057887.85
Branch                     24359201.21
Call Center                17364288.37
Web                        12697632.90


Analysis and Marketing Insights
Based on the resulting table, here are the key takeaways for the department:

Dominant Channel: Typically, Agent or Branch tends to lead in revenue. If the Agent channel has the highest sum, it suggests that personalized, human-led sales are the most effective at securing high-value long-term contracts.

Web vs. Call Center: If the Web channel shows significant revenue but lower costs than a Call Center, the marketing team should consider shifting more budget toward digital ads (PPC/SEO) to drive traffic to the site, as the ROI (Return on Investment) would be higher.

Mail Efficiency: If Mail is at the bottom (which is common), it may indicate that traditional direct mail is becoming obsolete for your demographic. However, if the average revenue per customer is high even if the total is low, it might still be a useful channel for "VIP" or luxury segments.



In [ ]:
import pandas as pd

# 1. Create a pivot table for Average CLV
# Index: Education level (rows)
# Columns: Gender (columns)
# Values: Customer Lifetime Value (aggregated by mean)
clv_pivot = df_marketing.pivot_table(
    index='education', 
    columns='gender', 
    values='customer_lifetime_value', 
    aggfunc='mean'
).round(2)

# Display the resulting pivot table
print("Average CLV by Education and Gender:")
print(clv_pivot)

To analyze the relationship between demographics and long-term profitability, we will create a multi-dimensional pivot table. This allows us to see how Gender and Education interact to influence the Customer Lifetime Value (CLV).

Following your coding standards for precise naming and clean structure, here is the implementation:

Python
import pandas as pd

# 1. Create a pivot table for Average CLV
# Index: Education level (rows)
# Columns: Gender (columns)
# Values: Customer Lifetime Value (aggregated by mean)
clv_pivot = df_marketing.pivot_table(
    index='education', 
    columns='gender', 
    values='customer_lifetime_value', 
    aggfunc='mean'
).round(2)

# Display the resulting pivot table
print("Average CLV by Education and Gender:")
print(clv_pivot)
Analysis and Strategic Insights
By examining the intersections of these two categories, the marketing department can draw several high-level conclusions:

1. The "Education Premium"
Finding: Usually, customers with higher education levels (Master or PhD) don't necessarily have a significantly higher CLV than those with a Bachelor's degree in insurance. In fact, sometimes "High School or Below" shows a high CLV due to long-term loyalty or specific policy types.

Marketing Action: If the data shows that Master/PhD holders have lower-than-expected CLV, the team should investigate if they are more likely to "churn" (switch companies) for better rates, suggesting a need for a loyalty program specifically for educated professionals.

2. Gender Parity Check
Finding: If the values between F and M are within a 5% margin across all education levels, it indicates that gender is not a strong predictor of total value.

Marketing Action: If gender is neutral, marketing creative (ads/emails) should focus more on lifestyle or education-based messaging rather than gender-specific campaigns.

3. High-Value Segments (The "Sweet Spot")
Finding: Look for the highest number in the table (for example, "Doctor" + "F"). This cell represents your most valuable demographic "persona."

Marketing Action: Use this persona to build Lookalike Audiences on social media platforms to acquire new customers who share these exact characteristics, as they are statistically likely to bring the most value to the company over time.